<a href="https://colab.research.google.com/github/tabeernaimal-svg/ai-research-paper-assistant/blob/main/AI_Research_Paper_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Research Paper Assistant

An NLP-based research paper assistant that uses sentence embeddings and FAISS similarity search to retrieve relevant information from a research paper.

## Project Overview

This project demonstrates a retrieval-based approach for interacting with research papers. A PDF research paper is converted into text, cleaned, divided into sentence-based chunks, transformed into vector embeddings, and indexed using FAISS.

When a user asks a question, the system converts the question into an embedding and retrieves the most semantically relevant passages from the paper.

## Technologies Used

- Python
- Sentence Transformers
- FAISS
- NumPy
- PyPDF
- Google Colab

## Pipeline

PDF → Text Extraction → Text Cleaning → Sentence Chunking → Embeddings → FAISS Index → Semantic Retrieval

## Paper Used

**A Survey of Large Language Models**

Wayne Xin Zhao et al.


## Project Workflow

The assistant follows this pipeline:

**PDF → Text Extraction → Text Cleaning → Sentence Chunking → Embeddings → FAISS Index → Semantic Retrieval**

### How It Works

1. A research paper is loaded from a PDF.
2. The text is extracted and cleaned.
3. The paper is divided into manageable sentence-based chunks.
4. Each chunk is converted into a numerical embedding using Sentence Transformers.
5. FAISS indexes the embeddings for efficient similarity search.
6. A user's question is also converted into an embedding.
7. The system retrieves the most semantically relevant passages from the paper.

## Project Objective

The goal of this project is to build a simple AI-assisted system that can understand a user's question and retrieve relevant evidence from a research paper using semantic similarity.

Instead of relying only on keyword matching, the system uses sentence embeddings to capture the meaning of the question and the paper's content.

This project focuses on the retrieval component of a Retrieval-Augmented Generation (RAG) system.

In [1]:
print("AI Research Paper Assistant - Ready!")

AI Research Paper Assistant - Ready!


## 1. Setup and Dependencies

The project uses Python libraries for PDF processing, sentence embeddings, numerical computation, and vector similarity search.


In [2]:
!pip install -q sentence-transformers pypdf faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 57.8 MB/s eta 0:00:00


## 2. Import Libraries

The required Python libraries are imported for PDF text extraction, numerical processing, sentence embeddings, and FAISS-based similarity search.

In [3]:
import os
import numpy as np
import pandas as pd

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss

## 3. Load and Extract the Research Paper

The research paper is loaded from a PDF file and its text is extracted page by page using PyPDF.

In [5]:
pdf_path = "/content/2303.18223v19.pdf"

reader = PdfReader(pdf_path)

text = ""

for page in reader.pages:
    text += page.extract_text() + "\n"

print("Number of pages:", len(reader.pages))
print("Characters extracted:", len(text))
print("\nFirst 2000 characters:\n")
print(text[:2000])

Number of pages: 144
Characters extracted: 861670

First 2000 characters:

1
A Survey of Large Language Models
Wayne Xin Zhao, Kun Zhou*, Junyi Li*, Tianyi Tang, Xiaolei Wang, Yupeng Hou, Yingqian Min, Beichen
Zhang, Junjie Zhang, Zican Dong, Yifan Du, Chen Y ang, Yushuo Chen, Zhipeng Chen, Jinhao Jiang,
Ruiyang Ren, Yifan Li, Xinyu Tang, Zikang Liu, Peiyu Liu, Jian-Yun Nie and Ji-Rong Wen†
Abstract—Ever since the Turing Test was proposed in the 1950s, humans have explored the mastering of language intelligence
by machine. Language is essentially a complex, intricate system of human expressions governed by grammatical rules. It poses a
significant challenge to develop capable artificial intelligence (AI) algorithms for comprehending and grasping a language. As a major
approach,language modelinghas been widely studied for language understanding and generation in the past two decades, evolving
from statistical language models to neural language models. Recently, pre-trained language mode

## 4. Clean the Extracted Text

The extracted paper text is cleaned by removing the reference section so that the retrieval system focuses on the main content of the research paper.

In [6]:
# Find the beginning of the reference section
reference_markers = [
    "\nReferences\n",
    "\nREFERENCES\n",
    "References"
]

reference_position = None

for marker in reference_markers:
    position = text.find(marker)
    if position != -1:
        reference_position = position
        break

if reference_position:
    clean_text = text[:reference_position]
else:
    clean_text = text

print("Original characters:", len(text))
print("Cleaned characters:", len(clean_text))
print("\nEnd of cleaned text:\n")
print(clean_text[-1000:])

Original characters: 861670
Cleaned characters: 602047

End of cleaned text:

 new contents are completed by a number of
volunteers in our team. Here, we add a special part to thank
all the students who have worked very hard on this part
(also including the ones on our author list).
Contribution on Experiments.We would like to sincerely
thank the following people for their hard work involved in
experiments shown in Table 16.
•Xiaoxue Cheng: implement the experiments for evalu-
ation on Language Generation and HaluEval tasks.
•Yuhao Wang: implement the experiments for evalua-
tion on interaction with environment tasks.
•Bowen Zheng: implement the experiments for evalua-
tion on tool manipulation tasks.
Contribution on Tips.We list the following guys for their
contributions on the corresponding numbers of provided
tips for designing prompts in Table 12.
•Xiaolei Wang: T3, O3
•Beichen Zhang: D2, D5
•Zhipeng Chen: D3, D4
•Junjie Zhang: D6
•Bowen Zheng: D7
•Zican Dong: D8
•Xinyu Tang: C2
•Y

## 5. Split the Paper into Sentence-Based Chunks

The cleaned paper is normalized and divided into manageable sentence-based chunks. This allows the retrieval system to search smaller sections of the paper rather than the entire document at once.


In [19]:
import re

normalized_text = re.sub(r'\s+', ' ', clean_text).strip()

# Remove the paper title, author information, and abstract
abstract_start = normalized_text.find("Abstract—")

if abstract_start != -1:
    body_start = normalized_text.find("1. INTRODUCTION", abstract_start)

    if body_start != -1:
        normalized_text = normalized_text[body_start:]

sentences = re.split(r'(?<=[.!?])\s+', normalized_text)

sentence_chunks = []
current_chunk = ""

for sentence in sentences:
    if len(current_chunk) + len(sentence) <= 1800:
        current_chunk += sentence + " "
    else:
        if current_chunk:
            sentence_chunks.append(current_chunk.strip())
        current_chunk = sentence + " "

if current_chunk:
    sentence_chunks.append(current_chunk.strip())
# Remove the initial title/abstract chunk from the retrieval corpus
sentence_chunks = sentence_chunks[1:]

print("Number of sentence chunks:", len(sentence_chunks))
print("\nFirst sentence chunk:\n")
print(sentence_chunks[0])

Number of sentence chunks: 349

First sentence chunk:

To discriminate the language models in different parameter scales, the research community has coined the termlarge language models (LLM)for the PLMs of significant size (e.g., containing tens or hundreds of billions of parameters). Recently, the research on LLMs has been largely advanced by both academia and industry, and a remarkable progress is the launch of ChatGPT (a powerful AI chatbot developed based on LLMs), which has attracted widespread attention from society. The technical evolution of LLMs has been making an important impact on the entire AI community, which would revolutionize the way how we develop and use AI algorithms. Considering this rapid technical progress, in this survey, we review the recent advances of LLMs by introducing the background, key findings, and mainstream techniques. In particular, we focus on four major aspects of LLMs, namely pre-training, adaptation tuning, utilization, and capacity evaluation. 

## 6. Generate Sentence Embeddings

Each sentence chunk is converted into a numerical vector using the `all-MiniLM-L6-v2` Sentence Transformer model.

These embeddings represent the semantic meaning of the text and allow the system to compare the meaning of a user's question with the content of the paper.

In [20]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully!


## 7. Create Embeddings for the Paper

The sentence chunks are passed through the embedding model to create a vector representation for each chunk.

These vectors will later be stored in a FAISS index for semantic similarity search.

In [21]:
sentence_embeddings = embedding_model.encode(
    sentence_chunks,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Embedding shape:", sentence_embeddings.shape)

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Embedding shape: (349, 384)


## 8. Build the FAISS Vector Index

FAISS is used to store the sentence embeddings and perform fast similarity searches.

The index uses inner-product similarity with normalized embeddings, which is equivalent to cosine similarity.

In [22]:
dimension = sentence_embeddings.shape[1]

sentence_index = faiss.IndexFlatIP(dimension)

sentence_index.add(
    np.array(sentence_embeddings).astype("float32")
)

print("Vectors stored:", sentence_index.ntotal)

Vectors stored: 349


[Markdown]
## 9. Semantic Retrieval

When a user asks a question, the question is converted into an embedding
and compared with the embeddings stored in the FAISS index.

The system then returns the most semantically similar passages from the
research paper.

        ↓

[Code]
def retrieve_answer(question, k=2):
    ...

In [23]:
def retrieve_answer(question, k=2):
    question_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    )

    distances, indices = sentence_index.search(
        np.array(question_embedding).astype("float32"),
        k
    )

    results = [sentence_chunks[i] for i in indices[0]]

    return results

In [24]:
question = "What special abilities do large language models exhibit?"

results = retrieve_answer(question, k=3)

print("Question:", question)
print("\nRelevant information from the paper:\n")

for i, result in enumerate(results, 1):
    print(f"\n--- Retrieved Passage {i} ---\n")
    print(result[:1200])

Question: What special abilities do large language models exhibit?

Relevant information from the paper:


--- Retrieved Passage 1 ---

With instruction tuning, LLMs are enabled to follow the task instructions for new tasks without using explicit examples, thus having an improved generalization ability. According to the experiments in [67], instruction-tuned LaMDA-PT [68] started to significantly outperform the untuned one on unseen tasks when the model size reached 68B, but not for 8B or smaller model sizes. A recent study [69] found that a model size of 62B is at least required for PaLM to perform well on various tasks in four evaluation benchmarks (i.e.,MMLU, BBH, TyDiQA and MGSM), though a much smaller size might suffice for some specific tasks (e.g.,MMLU). •Step-by-step reasoning.For small language models, it is usually difficult to solve complex tasks that involve 8. It is difficult to accurately examine the critical size for emergent abilities of LLMs (i.e.,the minimum size to p

## 10. Research Paper Assistant

The `paper_assistant()` function provides a simple interface for asking questions about the research paper.

It retrieves the most relevant passages and displays them as evidence, allowing the user to inspect the original information instead of relying on unsupported generated answers.

In [25]:
def paper_assistant(question, k=3):
    question_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    )

    distances, indices = sentence_index.search(
        np.array(question_embedding).astype("float32"),
        k
    )

    print("Question:", question)
    print("\nRelevant information from the paper:\n")

    for rank, idx in enumerate(indices[0], 1):
        print(f"\n--- Evidence {rank} ---")
        print(sentence_chunks[idx][:800])

## 11. Final Demonstration

The completed assistant is tested with a research question about the special abilities of large language models.

The system retrieves relevant evidence from the paper and presents the passages for inspection.

In [26]:
paper_assistant(
    "What special abilities do large language models exhibit?"
)

Question: What special abilities do large language models exhibit?

Relevant information from the paper:


--- Evidence 1 ---
With instruction tuning, LLMs are enabled to follow the task instructions for new tasks without using explicit examples, thus having an improved generalization ability. According to the experiments in [67], instruction-tuned LaMDA-PT [68] started to significantly outperform the untuned one on unseen tasks when the model size reached 68B, but not for 8B or smaller model sizes. A recent study [69] found that a model size of 62B is at least required for PaLM to perform well on various tasks in four evaluation benchmarks (i.e.,MMLU, BBH, TyDiQA and MGSM), though a much smaller size might suffice for some specific tasks (e.g.,MMLU). •Step-by-step reasoning.For small language models, it is usually difficult to solve complex tasks that involve 8. It is difficult to accurately examine the critical

--- Evidence 2 ---
Early lan- guage models mainly aim to model and gener